# 02 — Coordinate Projection

## The Problem with Angular Coordinates

Latitude and longitude are angular measurements, not distances. One degree of longitude at 80°N spans approximately 19 km of ground, while the same degree at the equator spans approximately 111 km. If we tried to build a uniform-area grid directly in (lat, lon) space, tiles near the poles would be far smaller in ground area than tiles at the equator — making the 250 m tile size meaningless at high latitudes.

## Solution: Web Mercator (EPSG:3857)

The Web Mercator projection maps (lat, lon) to (x, y) in metres using:

$$x = R \cdot \lambda$$

$$y = R \cdot \ln\!\left(\tan\!\left(\frac{\pi}{4} + \frac{\phi}{2}\right)\right)$$

where $R$ = 6 378 137 m (WGS84 equatorial radius), $\lambda$ = longitude in radians, $\phi$ = latitude in radians.

The logarithmic term — the inverse Gudermannian — makes the projection **conformal** (locally angle-preserving), which is why it is used in web mapping. However, it also compresses high-latitude areas, causing **scale distortion** (a 250 m Mercator tile covers more real ground near the poles).

The projection is undefined at the poles because $\tan(\pi/2)$ diverges; by convention, latitudes are clamped to ±85.05°.

<div style="background:#f5faf9;border:1px solid #b8ddd8;border-radius:8px;padding:12px 14px 14px;margin:10px 0 22px;font-family:sans-serif;">
<div style="font-size:11px;color:#5a9e99;margin-bottom:10px;font-style:italic;">Pipeline step 1 of 4 — Web Mercator projection</div>
<div style="display:flex;align-items:stretch;">
    <div style="background:#2a9d8f;color:white;clip-path:polygon(0 0, calc(100% - 22px) 0, 100% 50%, calc(100% - 22px) 100%, 0 100%);padding:10px 16px 10px 16px;margin-left:0;position:relative;z-index:1;min-width:115px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB02</div><div style="font-weight:700;font-size:13px;">① Project</div></div>
    <div style="background:#dff0ee;color:#3d7a71;clip-path:polygon(22px 0, calc(100% - 22px) 0, 100% 50%, calc(100% - 22px) 100%, 22px 100%, 0 50%);padding:10px 16px 10px 36px;margin-left:-21px;position:relative;z-index:2;min-width:115px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB03</div><div style="font-weight:500;font-size:13px;">② Snap+Shuffle</div></div>
    <div style="background:#dff0ee;color:#3d7a71;clip-path:polygon(22px 0, calc(100% - 22px) 0, 100% 50%, calc(100% - 22px) 100%, 22px 100%, 0 50%);padding:10px 16px 10px 36px;margin-left:-21px;position:relative;z-index:3;min-width:115px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB04</div><div style="font-weight:500;font-size:13px;">③ Lock</div></div>
    <div style="background:#dff0ee;color:#3d7a71;clip-path:polygon(22px 0, 100% 0, 100% 100%, 22px 100%, 0 50%);padding:10px 16px 10px 36px;margin-left:-21px;position:relative;z-index:4;min-width:115px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB05</div><div style="font-weight:500;font-size:13px;">④ Wobble</div></div>
</div>
</div>

## Learning Objectives

By the end of this notebook you will be able to:

1. **Recall** the Web Mercator forward and inverse projection formulae and the role of the Earth radius constant.
2. **Describe** why the logarithmic term in the Mercator formula causes scale distortion that increases toward the poles.
3. **Calculate** projected metre coordinates for each of the 8 John Snow pump locations and verify round-trip fidelity to 1 × 10⁻¹⁰ degrees.
4. **Examine** the scale-distortion chart and infer how distortion at latitude 51 °N affects tile-area uniformity.
5. **Justify** why metre-based tile indexing is more semantically consistent than degree-based indexing for a global dataset.

In [1]:
import math
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from map_encryption import _project, _unproject, _R_EARTH, _HALF_WORLD, SchemeParams

bin_size_m = SchemeParams().bin_size_m
M = 2 * int(math.floor(_HALF_WORLD / bin_size_m)) + 1
print(f'Half-world extent: {_HALF_WORLD:.3f} m')
print(f'Bin size: {bin_size_m} m')
print(f'Total grid cells per axis: M = {M}')
print(f'Total cells: M^2 = {M**2:,}')

Half-world extent: 20037508.343 m
Bin size: 250 m
Total grid cells per axis: M = 160301
Total cells: M^2 = 25,696,410,601


In [2]:
import pandas as pd

pumps_df = pd.read_csv('data/pumps.csv')
pumps = list(zip(pumps_df['Street'], pumps_df['LAT'], pumps_df['LON']))

# John Snow identified the Broadwick Street pump as the outbreak source.
# All 8 pump locations fit within Soho — a tight ~1 km² area that exercises
# the projection at close range (verifying metre-level accuracy, not just global).

print(f'{"Pump (Street)":<22} {"Lat":>10} {"Lon":>10} {"x (m)":>15} {"y (m)":>15}')
print('-' * 78)
for name, lat, lon in pumps:
    x, y = _project(lat, lon)
    rlat, rlon = _unproject(x, y)
    err = max(abs(rlat - lat), abs(rlon - lon))
    assert err < 1e-10, f'Round-trip error {err:.2e} exceeds tolerance for {name}'
    print(f'{name:<22} {lat:>10.6f} {lon:>10.6f} {x:>15.3f} {y:>15.3f}')

print(f'\nAll {len(pumps)} pump round-trips verified (error < 1e-10 degrees).')


Pump (Street)                 Lat        Lon           x (m)           y (m)
------------------------------------------------------------------------------
Broadwick Street        51.513341  -0.136668      -15213.812     6712605.101
Kingly Street           51.513876  -0.139586      -15538.642     6712700.799
Ramillies Place         51.514906  -0.139671      -15548.105     6712885.044
Dean Street             51.512354  -0.131630      -14652.985     6712428.553
Rupert Street           51.512139  -0.133594      -14871.616     6712390.096
Bridle Lane             51.511542  -0.135919      -15130.434     6712283.311
Regent Street           51.510019  -0.133962      -14912.582     6712010.901
Warwick Street          51.511295  -0.138199      -15384.242     6712239.131

All 8 pump round-trips verified (error < 1e-10 degrees).


In [3]:
lats_range = np.linspace(-80, 80, 400)
y_vals = [_project(lat, 0)[1] for lat in lats_range]
scale_factors = [math.cos(math.radians(lat)) for lat in lats_range]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lats_range, y=y_vals, name='y_mercator (m)',
    line=dict(color='steelblue', width=2)
))
fig.add_trace(go.Scatter(
    x=lats_range, y=[s * max(y_vals) for s in scale_factors],
    name='scale factor × y_max (normalised)',
    line=dict(color='tomato', width=2, dash='dash'),
    yaxis='y2'
))
fig.update_layout(
    title='Web Mercator y-coordinate and scale factor vs latitude',
    xaxis_title='Latitude (degrees)',
    yaxis_title='y_mercator (m)',
    yaxis2=dict(title='Scale factor (cos lat)', overlaying='y', side='right'),
    template='plotly_white',
    height=400,
    legend=dict(x=0.02, y=0.98)
)
fig.show()

**Figure 2a.** Web Mercator scale factor (sec φ = 1/cos φ) plotted against latitude from 0° to 80°N, illustrating how the projected length of a 250 m tile expands toward the poles.

## Grid Dimensions

With `bin_size_m = 250` m and `_HALF_WORLD ≈ 20 037 508` m:

```
M = 2 * floor(20_037_508 / 250) + 1 = 160_301 cells per axis
Total cells = M² = 25_696_411_601 ≈ 25.7 billion
```

Each tile index (qx, qy) is a pair of integers in [-80150, +80150]. Even though the Mercator projection distorts scale at high latitudes, the **index space** is uniform: every (qx, qy) pair is exactly one 250 m Mercator-unit tile. The PRP shuffles these indices, so an attacker who sees encrypted (qxp, qyp) values cannot determine which region of the globe they came from.

In [4]:
import pandas as pd
import folium

rows = []
for name, lat, lon in pumps:
    x, y = _project(lat, lon)
    rlat, rlon = _unproject(x, y)
    rows.append({'pump': name, 'lat': lat,  'lon': lon,  'type': 'original'})
    rows.append({'pump': name, 'lat': rlat, 'lon': rlon, 'type': 'recovered'})

df = pd.DataFrame(rows)

# All 8 pumps are within Soho — zoom 14 shows the tight geographic cluster.
# 'recovered' pins overlay 'original' exactly, confirming lossless round-trip.
colors = {'original': 'steelblue', 'recovered': 'tomato'}
m = folium.Map(location=[51.513, -0.136], zoom_start=14, tiles='cartodbpositron')
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=7,
        color=colors[row['type']],
        fill=True,
        fill_color=colors[row['type']],
        fill_opacity=0.8,
        tooltip=f"{row['pump']} — {row['type']}",
    ).add_to(m)
m


**Figure 2b.** Folium map of the eight John Snow pump locations projected into Web Mercator and centred on the Broadwick Street pump in Soho, London.

In [5]:
lat_samples = [0, 20, 40, 60, 80]
ground_km = [250 / math.cos(math.radians(lat)) / 1000 for lat in lat_samples]

fig = px.bar(
    x=[str(lat) + '°' for lat in lat_samples],
    y=ground_km,
    labels={'x': 'Latitude', 'y': 'Effective ground size (km)'},
    title='Effective ground size of a 250 m Mercator tile by latitude',
    template='plotly_white',
    height=400,
    color=ground_km,
    color_continuous_scale='Oranges'
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

**Figure 2c.** Bar chart of tile ground size (km) at eight reference latitudes, quantifying the monotonic increase in physical distance represented by a 250 m projected tile as latitude rises toward the poles.

## What the Projection Gives — and Does Not Give

**Gives:** uniform index space (every cell is 250 m × 250 m in Mercator units), global coverage (±85.05° latitude), full invertibility, and conformality (locally angle-preserving, useful for short-range distance calculations).

**Does not give:** equal-area (a 250 m tile covers more ground at high latitudes), accuracy above ±85.05° (pole singularity), or metric accuracy at continental scales.

NB03 builds the tile grid and the Feistel PRP on top of this projection.

## References

- **Snow, J.** (1855). *On the Mode of Communication of Cholera* (2nd ed.). Churchill, London. — Source of the 1854 Soho cholera death and pump location dataset used throughout these notebooks.
- **Snyder, J.P.** (1987). *Map Projections — A Working Manual.* USGS Professional Paper 1395. — Reference for the Web Mercator projection used in NB02.

## Glossary

| Term | Definition |
|------|-----------|
| **Web Mercator** | A cylindrical map projection (EPSG:3857) that maps (lat, lon) to (x, y) in metres; used by most web mapping services. |
| **Forward projection** | Converting geographic (lat, lon) degrees to projected (x, y) metres via the Mercator formula. |
| **Inverse projection** | Recovering (lat, lon) from projected (x, y) metres; the exact mathematical inverse of the forward projection. |
| **Scale distortion** | The factor by which Mercator stretches east-west distances at a given latitude; equals `1 / cos(lat)`, reaching infinity at the poles. |
| **Pole singularity** | The point at ±90° latitude where the Mercator projection is undefined because `cos(90°) = 0`. |
| **Inverse Gudermannian** | The mathematical function `ln(tan(π/4 + lat/2))` that appears in the Mercator y-coordinate formula; maps latitude to a linear scale. |
| **Round-trip fidelity** | The property that `unproject(project(lat, lon)) == (lat, lon)` to within floating-point precision (< 1 × 10⁻¹⁰ degrees here). |
| **Projected coordinates** | The (x, y) values in metres on the Mercator plane; used for tile indexing because they have uniform spacing regardless of longitude. |